In [ ]:
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import Namespace
import requests
import time

WDT = Namespace("http://www.wikidata.org/prop/direct/")
GEO = Namespace("http://www.opengis.net/ont/geosparql#")

WD_HTTP = "http://www.wikidata.org/entity/"
WD_HTTPS = "https://www.wikidata.org/entity/"
SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
USER_AGENT = "Wikidata-Coordinate-Enricher/6.0"

TRUSTED_LOCATION_PROPERTIES = {
    str(WDT.P17),    # country
    str(WDT.P27),    # country of citizenship
    str(WDT.P276),   # location
    str(WDT.P1071),  # location of creation
    str(WDT.P19),    # place of birth
    str(WDT.P20),    # place of death
    str(WDT.P937),   # work location
    str(WDT.P131),   # located in administrative territorial entity
    str(WDT.P551),   # residence
    str(WDT.P740),   # location of formation
    str(WDT.P159),   # headquarters location
}


def normalize_wd_iri(iri: str) -> str:
    return iri.replace(WD_HTTPS, WD_HTTP)


def is_wikidata_entity_iri(iri: str) -> bool:
    return iri.startswith(WD_HTTP) or iri.startswith(WD_HTTPS)


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


def run_sparql(query: str, timeout: int = 30):
    r = requests.get(
        SPARQL_ENDPOINT,
        params={"query": query, "format": "json"},
        headers={"User-Agent": USER_AGENT},
        timeout=timeout
    )
    r.raise_for_status()
    return r.json()


# -----------------------------------------
# 1. Load dataset
# -----------------------------------------
g = Graph()
g.parse("data10.ttl", format="turtle")
print("Loaded triples:", len(g))

# -----------------------------------------
# 2. Collect Wikidata entities already present anywhere in graph
# -----------------------------------------
wikidata_entities = set()

for s, p, o in g:
    for node in (s, o):
        if isinstance(node, URIRef):
            iri = str(node)
            if is_wikidata_entity_iri(iri):
                wikidata_entities.add(normalize_wd_iri(iri))

print("Wikidata entities already in graph:", len(wikidata_entities))

# -----------------------------------------
# 3. Collect linked location entities from LOCAL RDF predicates
# -----------------------------------------
linked_entities = set()

for s, p, o in g:
    if not isinstance(p, URIRef) or not isinstance(o, URIRef):
        continue

    pred = str(p)
    obj = str(o)

    if pred in TRUSTED_LOCATION_PROPERTIES and is_wikidata_entity_iri(obj):
        linked_entities.add(normalize_wd_iri(obj))

print("Linked location entities found locally:", len(linked_entities))

# Optional debug
test_q = "http://www.wikidata.org/entity/Q256004"
print("Q256004 found among local linked entities?", test_q in linked_entities)

# -----------------------------------------
# 4. Candidate entities for coordinate lookup
# -----------------------------------------
candidate_entities = sorted(linked_entities)
print("Total candidate entities for coordinate lookup:", len(candidate_entities))

# -----------------------------------------
# 5. Query Wikidata API for coordinates
# -----------------------------------------

WIKIDATA_API = "https://www.wikidata.org/w/api.php"

coords = {}


def get_coordinates_from_api(batch, max_retries=5):
    qids = [
        iri.rsplit("/", 1)[-1]
        for iri in batch
    ]

    params = {
        "action": "wbgetentities",
        "ids": "|".join(qids),
        "props": "claims",
        "format": "json",
        "formatversion": 2
    }

    headers = {
        "User-Agent": "Wikidata-Coordinate-Enricher/6.0"
    }

    for attempt in range(max_retries):
        try:
            r = requests.get(
                WIKIDATA_API,
                params=params,
                headers=headers,
                timeout=60
            )

            if r.status_code == 429:
                retry_after = r.headers.get("Retry-After")

                try:
                    wait = int(retry_after)
                except (TypeError, ValueError):
                    wait = 20 * (attempt + 1)

                print(f"Wikidata API rate limited. Waiting {wait}s...")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r.json()

        except requests.exceptions.RequestException as e:
            wait = 10 * (attempt + 1)
            print(f"Wikidata API error: {e}")
            print(f"Waiting {wait}s...")
            time.sleep(wait)

    return None


total_batches = (len(candidate_entities) + 49) // 50

for batch_number, batch in enumerate(
    chunks(candidate_entities, 50),
    start=1
):
    print(
        f"Coordinate batch {batch_number}/{total_batches} "
        f"({len(batch)} entities)"
    )

    data = get_coordinates_from_api(batch)

    if not data:
        print("Batch failed.")
        continue

    for qid, entity in data.get("entities", {}).items():

        if not isinstance(entity, dict):
            continue

        claims = entity.get("claims", {})
        p625 = claims.get("P625", [])

        if not p625:
            continue

        for statement in p625:
            try:
                snak = statement["mainsnak"]

                if snak.get("snaktype") != "value":
                    continue

                value = snak["datavalue"]["value"]

                latitude = value["latitude"]
                longitude = value["longitude"]

                iri = WD_HTTP + qid

                coords[iri] = f"Point({longitude} {latitude})"

                break

            except (KeyError, TypeError):
                continue

    print("Coordinates collected so far:", len(coords))

    time.sleep(1.0)

print("Coordinates retrieved:", len(coords))
print(
    "Q256004 got coordinates?",
    test_q in coords,
    coords.get(test_q)
)

# -----------------------------------------
# 6. Add coordinates only to the entity that has them
# -----------------------------------------
added = 0

for entity, coord in coords.items():
    http_version = entity
    https_version = entity.replace("http://", "https://")

    triple_http = (URIRef(http_version), WDT.P625, Literal(coord, datatype=GEO.wktLiteral))
    triple_https = (URIRef(https_version), WDT.P625, Literal(coord, datatype=GEO.wktLiteral))

    if triple_http not in g:
        g.add(triple_http)
        added += 1

    if triple_https not in g:
        g.add(triple_https)
        added += 1

print("Coordinate triples added:", added)

# -----------------------------------------
# 7. Save enriched dataset
# -----------------------------------------
output_file = "dataset11.ttl"
g.serialize(output_file, format="turtle")
print(f"Saved as {output_file}")